In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import pickle
import os
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import issparse

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/fake-news-detection/dataset/WELFake_Dataset.csv')
df = df.dropna(subset=['title', 'text', 'label'])
df['content'] = df['title'] + " " + df['text']
df = df[['content', 'label']].reset_index(drop=True)

print(df['label'].value_counts())
print(df.shape)

In [ ]:
#Split Data
X = df['content'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
#  Load Tokenizer & Padding
MAX_LEN = 300
VOCAB_SIZE = 50000

with open('/content/drive/MyDrive/fake-news-detection/tokenizer/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

print("✅ Tokenizer loaded!")

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, truncating='post', padding='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, truncating='post', padding='post')

print(f"X_train pad shape: {X_train_pad.shape}")
print(f"X_test pad shape:  {X_test_pad.shape}")

In [ ]:
# TF-IDF Features
TFIDF_MAX_FEATURES = 5000

tfidf = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES)
X_train_tfidf = tfidf.fit_transform(X_train).toarray().astype(np.float32)
X_test_tfidf  = tfidf.transform(X_test).toarray().astype(np.float32)

print(f"X_train tfidf shape: {X_train_tfidf.shape}")
print(f"X_test tfidf shape:  {X_test_tfidf.shape}")

# Simpan TF-IDF vectorizer
with open('/content/drive/MyDrive/fake-news-detection/tokenizer/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("✅ TF-IDF vectorizer saved!")

In [ ]:
#  Custom Attention Layer
class AttentionLayer(layers.Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name='attention_weight',
            shape=(input_shape[-1], 1),
            initializer='random_normal',
            trainable=True
        )
        self.b = self.add_weight(
            name='attention_bias',
            shape=(input_shape[1], 1),
            initializer='zeros',
            trainable=True
        )
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        e = tf.nn.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = x * a
        return tf.reduce_sum(output, axis=1)

    def get_config(self):
        return super(AttentionLayer, self).get_config()

In [ ]:
# Bangun Model Hybrid (Functional API)
EMBED_DIM = 128

# ── Branch 1: Word Embedding + CNN + BiLSTM ───────────
input_seq = layers.Input(shape=(MAX_LEN,), name='input_sequence')
x = layers.Embedding(VOCAB_SIZE, EMBED_DIM, name='embedding')(input_seq)
x = layers.Conv1D(128, kernel_size=3, activation='relu', name='conv1d')(x)
x = layers.MaxPooling1D(pool_size=2, name='maxpool')(x)
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True), name='bilstm')(x)
x = layers.Dropout(0.3, name='dropout_seq')(x)
x = AttentionLayer(name='attention')(x)

# ── Branch 2: TF-IDF ──────────────────────────────────
input_tfidf = layers.Input(shape=(TFIDF_MAX_FEATURES,), name='input_tfidf')
t = layers.Dense(256, activation='relu', name='tfidf_dense_1')(input_tfidf)
t = layers.Dropout(0.3, name='dropout_tfidf')(t)
t = layers.Dense(128, activation='relu', name='tfidf_dense_2')(t)

# ── Gabungkan kedua branch ────────────────────────────
combined = layers.Concatenate(name='concatenate')([x, t])
z = layers.Dense(128, activation='relu', name='combined_dense')(combined)
z = layers.Dropout(0.3, name='dropout_combined')(z)
outputs = layers.Dense(1, activation='sigmoid', name='output')(z)

model = Model(
    inputs=[input_seq, input_tfidf],
    outputs=outputs,
    name='CNN_BiLSTM_TFIDF_Hybrid'
)
model.summary()

In [ ]:
# Callback Config
EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

ES_PATIENCE = 2
ES_MIN_DELTA = 1e-4

LR_PATIENCE = 2
LR_FACTOR = 0.3
LR_MIN = 1e-5

best_val_loss = np.inf
patience_counter = 0
lr_patience_counter = 0
current_lr = LEARNING_RATE

optimizer = tf.keras.optimizers.Adam(learning_rate=current_lr)
loss_fn = tf.keras.losses.BinaryCrossentropy()

log_dir = '/content/drive/MyDrive/fake-news-detection/logs/cnn_bilstm'
os.makedirs(log_dir, exist_ok=True)
writer = tf.summary.create_file_writer(log_dir)

checkpoint_path = '/content/drive/MyDrive/fake-news-detection/models/best_model_cnn_bilstm.keras'
os.makedirs('/content/drive/MyDrive/fake-news-detection/models', exist_ok=True)

print("✅ Callback config ready!")

In [ ]:
# Training Loop GradientTape
# Dataset dengan 2 input
train_dataset = tf.data.Dataset.from_tensor_slices(
    ((X_train_pad, X_train_tfidf), y_train)
).shuffle(10000).batch(BATCH_SIZE)

test_dataset = tf.data.Dataset.from_tensor_slices(
    ((X_test_pad, X_test_tfidf), y_test)
).batch(BATCH_SIZE)

train_acc_metric = tf.keras.metrics.BinaryAccuracy()
val_acc_metric   = tf.keras.metrics.BinaryAccuracy()

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

for epoch in range(EPOCHS):
    print(f"\n🔄 Epoch {epoch+1}/{EPOCHS} | LR: {current_lr:.6f}")
    train_acc_metric.reset_states()
    val_acc_metric.reset_states()
    epoch_loss = []

    # ── Training ──────────────────────────────────
    for (x_seq, x_tfidf), y_batch in train_dataset:
        with tf.GradientTape() as tape:
            preds = model([x_seq, x_tfidf], training=True)
            loss  = loss_fn(y_batch, preds)
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        train_acc_metric.update_state(y_batch, preds)
        epoch_loss.append(loss.numpy())

    # ── Validation ────────────────────────────────
    val_loss_list = []
    for (x_seq, x_tfidf), y_batch in test_dataset:
        val_preds = model([x_seq, x_tfidf], training=False)
        val_loss  = loss_fn(y_batch, val_preds)
        val_acc_metric.update_state(y_batch, val_preds)
        val_loss_list.append(val_loss.numpy())

    avg_loss     = np.mean(epoch_loss)
    avg_val_loss = np.mean(val_loss_list)
    avg_acc      = train_acc_metric.result().numpy()
    avg_val_acc  = val_acc_metric.result().numpy()

    history['loss'].append(avg_loss)
    history['accuracy'].append(avg_acc)
    history['val_loss'].append(avg_val_loss)
    history['val_accuracy'].append(avg_val_acc)

    print(f"📊 loss: {avg_loss:.4f} | acc: {avg_acc:.4f} | val_loss: {avg_val_loss:.4f} | val_acc: {avg_val_acc:.4f}")

    # ── TensorBoard ───────────────────────────────
    with writer.as_default():
        tf.summary.scalar('loss',         avg_loss,     step=epoch)
        tf.summary.scalar('accuracy',     avg_acc,      step=epoch)
        tf.summary.scalar('val_loss',     avg_val_loss, step=epoch)
        tf.summary.scalar('val_accuracy', avg_val_acc,  step=epoch)

    # ── ModelCheckpoint ───────────────────────────
    if avg_val_loss < best_val_loss - ES_MIN_DELTA:
        print(f"💾 val_loss improved ({best_val_loss:.4f} → {avg_val_loss:.4f}), saving model...")
        best_val_loss = avg_val_loss
        patience_counter = 0
        lr_patience_counter = 0
        model.save(checkpoint_path)
    else:
        patience_counter += 1
        lr_patience_counter += 1
        print(f"⏳ No improvement. EarlyStopping: {patience_counter}/{ES_PATIENCE} | ReduceLR: {lr_patience_counter}/{LR_PATIENCE}")

    # ── ReduceLROnPlateau ─────────────────────────
    if lr_patience_counter >= LR_PATIENCE:
        new_lr = max(current_lr * LR_FACTOR, LR_MIN)
        if new_lr < current_lr:
            current_lr = new_lr
            optimizer.learning_rate.assign(current_lr)
            print(f"📉 ReduceLROnPlateau: LR diturunkan → {current_lr:.6f}")
        lr_patience_counter = 0

    # ── EarlyStopping ─────────────────────────────
    if patience_counter >= ES_PATIENCE:
        print(f"\n🛑 EarlyStopping triggered! Best val_loss: {best_val_loss:.4f}")
        break

print("\n✅ Training selesai!")

# Load best model
model = tf.keras.models.load_model(
    checkpoint_path,
    custom_objects={'AttentionLayer': AttentionLayer}
)
print("✅ Best model loaded!")

In [ ]:
# Visualisasi & Evaluasi
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['loss'],     label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_title('Loss - CNN + BiLSTM + TF-IDF Hybrid')
ax1.legend()

ax2.plot(history['accuracy'],     label='Train Accuracy')
ax2.plot(history['val_accuracy'], label='Val Accuracy')
ax2.set_title('Accuracy - CNN + BiLSTM + TF-IDF Hybrid')
ax2.legend()

plt.tight_layout()
plt.show()

y_pred = (model.predict([X_test_pad, X_test_tfidf]) >= 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=['FAKE', 'REAL']))

In [ ]:
# Simpan Model
model.save('/content/drive/MyDrive/fake-news-detection/models/model_cnn_bilstm.keras')
print("✅ Model CNN + BiLSTM saved!")